# Feature Level Embedding

embedding = MLP(concat(TPC_CNN, TPC_ESM_CNN, TPC_Mol2Vec, FP_MLP))

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.preprocessing import StandardScaler
import os
import random

# --------------------------
# 1 Fix random seed
# --------------------------
SEED = 42
#3
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# --------------------------
# 2 Config / Hyperparameters
# --------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 32
HIDDEN_DIM = 8
N_HEADS = 2
N_LAYERS = 1
LR = 1e-3
WEIGHT_DECAY = 1e-4
EPOCHS = 50
OUTPUT_DIR = "/kaggle/working/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --------------------------
# 3 Load data
# --------------------------
X_train = pd.read_csv("/kaggle/input/mapms-main-type1-unembed/M1_Train.csv").iloc[:, 1:].values
X_test  = pd.read_csv("/kaggle/input/mapms-main-type1-unembed/M1_Test.csv").iloc[:, 1:].values


y_train = [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
                  
y_test = [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

y_train = np.array(y_train)
y_test  = np.array(y_test)

# --------------------------
#  Scale features
# --------------------------
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

# --------------------------
# 4 Dataset
# --------------------------
from sklearn.model_selection import train_test_split

# Split train into train + validation
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=SEED, stratify=y_train
)

class TabDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_ds = TabDataset(X_tr, y_tr)
val_ds   = TabDataset(X_val, y_val)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

# --------------------------
# 5 MLP Model
# --------------------------
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, dropout=0.3):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x, return_feature=False):
        h = x
        for layer in self.model[:-1]:
            h = layer(h)
        if return_feature:
            return h
        logits = self.model[-1](h).squeeze(1)
        return logits

model = MLP(input_dim=X_train.shape[1], hidden_dim=HIDDEN_DIM, dropout=0.3).to(DEVICE)
criterion = nn.BCEWithLogitsLoss()
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

# --------------------------
# 6 Train loop with validation + early stopping
# --------------------------
best_val_loss = float('inf')
patience = 5
counter = 0

for epoch in range(EPOCHS):
    # Train
    model.train()
    epoch_loss = 0.0
    for x_batch, y_batch in train_loader:
        x_batch, y_batch = x_batch.to(DEVICE), y_batch.to(DEVICE)
        optimizer.zero_grad()
        logits = model(x_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * x_batch.size(0)
    epoch_loss /= len(train_loader.dataset)

    # Validation
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for x_batch, y_batch in val_loader:
            x_batch, y_batch = x_batch.to(DEVICE), y_batch.to(DEVICE)
            logits = model(x_batch)
            loss = criterion(logits, y_batch)
            val_loss += loss.item() * x_batch.size(0)
    val_loss /= len(val_loader.dataset)

    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {epoch_loss:.4f} | Val Loss: {val_loss:.4f}")

    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_model.pth")
        counter = 0
    else:
        counter += 1
        if counter >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break

# Load best model
model.load_state_dict(torch.load("best_model.pth"))

# --------------------------
# 7 Feature extraction
# --------------------------
def extract_feature(model, X, batch_size=64):
    model.eval()
    feats = []
    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            x_batch = torch.tensor(X[i:i+batch_size], dtype=torch.float32).to(DEVICE)
            h = model(x_batch, return_feature=True)
            feats.append(h.cpu().numpy())
    return np.vstack(feats)

Z_train = extract_feature(model, X_train)
Z_test  = extract_feature(model, X_test)

# --------------------------
# 8 Save features
# --------------------------
pd.DataFrame(Z_train).to_csv(os.path.join(OUTPUT_DIR, "EmbedSup4-2_main_train.csv"), index=True)
pd.DataFrame(Z_test).to_csv(os.path.join(OUTPUT_DIR, "EmbedSup4-2-main_test.csv"), index=True)

print("Feature extraction complete!")
print(f"Z_train shape: {Z_train.shape}")
print(f"Z_test  shape: {Z_test.shape}")
print(f"Saved CSV to {OUTPUT_DIR}")



# Classifier

In [ ]:
# Fully Connected Classifier
# ============================

import os
import math
import random
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_curve, auc, precision_recall_curve
from sklearn.model_selection import StratifiedKFold

from tensorflow.keras.models import Sequential, model_from_json
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam


# ============================
# 🔒 Fix random seed
# ============================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["TF_DETERMINISTIC_OPS"] = "1"


# -----------------------------
# Utility functions
# -----------------------------
def categorical_probas_to_classes(p):
    return np.argmax(p, axis=1)

def calculate_performance(test_num, pred_y, labels):
    tp = fp = tn = fn = 0
    for i in range(test_num):
        if labels[i] == 1:
            if pred_y[i] == 1: tp += 1
            else: fn += 1
        else:
            if pred_y[i] == 0: tn += 1
            else: fp += 1

    acc = (tp + tn) / test_num
    precision = tp / (tp + fp + 1e-6)
    npv = tn / (tn + fn + 1e-6)
    sensitivity = tp / (tp + fn + 1e-6)
    specificity = tn / (tn + fp + 1e-6)
    mcc = (tp * tn - fp * fn) / (
        math.sqrt((tp + fp)*(tp + fn)*(tn + fp)*(tn + fn)) + 1e-6
    )
    f1 = 2 * tp / (2 * tp + fp + fn + 1e-6)

    return acc, precision, npv, sensitivity, specificity, mcc, f1, tp, tn, fp, fn


# -----------------------------
# Model (DNN)
# -----------------------------
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam

def get_FC_model(input_dim, out_dim):
    model = Sequential([
        Dense(256, activation='relu', input_shape=(input_dim,),
              kernel_initializer='he_normal'),
        BatchNormalization(),
        Dropout(0.3),

        Dense(128, activation='relu',
              kernel_initializer='he_normal'),
        BatchNormalization(),
        Dropout(0.7),

        Dense(64, activation='relu',
              kernel_initializer='he_normal'),
        BatchNormalization(),
        Dropout(0.6),

        Dense(out_dim, activation='softmax')
    ])

    model.compile(
        optimizer=Adam(learning_rate=1e-3),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model



# -----------------------------
# Load data
# -----------------------------
train_path = '/kaggle/working/EmbedSup4-2_main_train.csv'
test_path  = '/kaggle/working/EmbedSup4-2-main_test.csv'




train_df = pd.read_csv(train_path)
test_df  = pd.read_csv(test_path)

X_train = train_df.iloc[:, 1:].values
X_test  = test_df.iloc[:, 1:].values

y_train = [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
                  
y_test = [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
y_train = np.array(y_train)
y_test  = np.array(y_test)



# -----------------------------
# Normalize (ถูกหลัก)
# -----------------------------
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

input_dim = X_train.shape[1]
out_dim = len(np.unique(y_train))


# -----------------------------
# Cross-validation
# -----------------------------
save_dir = f'./DNN_seed{SEED}/'
os.makedirs(save_dir, exist_ok=True)

sepscores_cnn = []

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss', factor=0.2, patience=5, min_lr=1e-6
)
early_stop = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)
callbacks = [reduce_lr, early_stop]

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)

# ===== ADD (for ProbFeat only) =====
oof_train_prob = np.zeros((len(X_train), out_dim))
test_prob_folds = []

for i, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    print(f"\n===== Fold {i} =====")

    model = get_FC_model(input_dim, out_dim)

    X_tr  = X_train[tr_idx]
    X_val = X_train[val_idx]
    y_tr  = to_categorical(y_train[tr_idx], out_dim)
    y_val = to_categorical(y_train[val_idx], out_dim)

    model.fit(
        X_tr, y_tr,
        validation_data=(X_val, y_val),
        epochs=50,
        batch_size=8,
        callbacks=callbacks,
        verbose=1
    )

    # ===== validation (เดิม) =====
    y_pred_prob = model.predict(X_val)
    y_pred = np.argmax(y_pred_prob, axis=1)

    acc, precision, npv, sensitivity, specificity, mcc, f1, tp, tn, fp, fn = \
        calculate_performance(len(y_pred), y_pred, y_train[val_idx])

    fpr, tpr, _ = roc_curve(y_train[val_idx], y_pred_prob[:, 1])
    roc_auc = auc(fpr, tpr)

    precision_curve, recall_curve, _ = precision_recall_curve(
        y_train[val_idx], y_pred_prob[:, 1]
    )
    au_pr = auc(recall_curve, precision_curve)

    sepscores_cnn.append(
        [acc, precision, npv, sensitivity, specificity,
         mcc, f1, roc_auc, au_pr, tp, tn, fp, fn]
    )

    # ===== ADD ONLY =====
    oof_train_prob[val_idx] = y_pred_prob
    test_prob_folds.append(model.predict(X_test))

    # Save model (เดิม)
    with open(f"{save_dir}/DNN_fold{i}.json", "w") as f:
        f.write(model.to_json())
    model.save_weights(f"{save_dir}/DNN_fold{i}.weights.h5")


cv_df = pd.DataFrame(
    sepscores_cnn,
    columns=['acc','precision','npv','sensitivity','specificity',
             'mcc','f1','roc_auc','au_pr','tp','tn','fp','fn']
)
cv_df.to_csv(f"{save_dir}/results_CV_DNNEmbedSup64-main.csv", index=False)


# -----------------------------
# Independent test (เดิมทั้งหมด)
# -----------------------------
sepscores_ind = []

for i in range(len(sepscores_cnn)):
    with open(f"{save_dir}/DNN_fold{i}.json", "r") as f:
        loaded_model = model_from_json(f.read())

    loaded_model.load_weights(f"{save_dir}/DNN_fold{i}.weights.h5")
    loaded_model.compile(
        loss='categorical_crossentropy',
        optimizer=Adam(),
        metrics=['accuracy']
    )

    y_score = loaded_model.predict(X_test)
    y_class = np.argmax(y_score, axis=1)

    acc, precision, npv, sensitivity, specificity, mcc, f1, tp, tn, fp, fn = \
        calculate_performance(len(y_class), y_class, y_test)

    fpr, tpr, _ = roc_curve(y_test, y_score[:, 1])
    roc_auc = auc(fpr, tpr)

    precision_curve, recall_curve, _ = precision_recall_curve(
        y_test, y_score[:, 1]
    )
    au_pr = auc(recall_curve, precision_curve)

    sepscores_ind.append(
        [acc, precision, npv, sensitivity, specificity,
         mcc, f1, roc_auc, au_pr, tp, tn, fp, fn]
    )

    print(f"Fold {i} test: MCC={mcc:.4f}, AUC={roc_auc:.4f}, AU_PR={au_pr:.4f}")

test_df_result = pd.DataFrame(
    sepscores_ind,
    columns=['acc','precision','npv','sensitivity','specificity',
             'mcc','f1','roc_auc','au_pr','tp','tn','fp','fn']
)
test_df_result.to_csv(f"{save_dir}/results_Independent_DNNEmbedSup64-main.csv", index=False)


# =============================
# ✅ SAVE ProbFeat (NEW ONLY)
# =============================
prob_train_df = pd.DataFrame(
    oof_train_prob,
    columns=[f'Prob_class{i}' for i in range(out_dim)]
)
prob_train_df.to_csv(f"{save_dir}/ProbFeat_train.csv", index=False)

avg_test_prob = np.mean(test_prob_folds, axis=0)

prob_test_df = pd.DataFrame(
    avg_test_prob,
    columns=[f'Prob_class{i}' for i in range(out_dim)]
)
prob_test_df.to_csv(f"{save_dir}/ProbFeat_test.csv", index=False)

print("✅ Saved ProbFeat_train.csv and ProbFeat_test.csv")


# ก่อน main หลัง alt

In [ ]:
# แบบ alt ไม่ต้องมี gpu
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.preprocessing import StandardScaler
import os
import random

# --------------------------
# 0️⃣ Fix random seed
# --------------------------
SEED = 5

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# --------------------------
# 1️⃣ Config / Hyperparameters
# --------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 32
HIDDEN_DIM = 8
N_HEADS = 2
N_LAYERS = 1
LR = 1e-3
WEIGHT_DECAY = 1e-4
EPOCHS = 50
OUTPUT_DIR = "/kaggle/working/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --------------------------
# 2️⃣ Load data
# --------------------------
X_train = pd.read_csv("/kaggle/input/mapms-type1-2-unembed/M1-alt_Train.csv").iloc[:, 1:].values
X_test  = pd.read_csv("/kaggle/input/mapms-type1-2-unembed/M1-alt_Test.csv").iloc[:, 1:].values


y_train = [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
                  
y_test = [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

y_train = np.array(y_train)
y_test  = np.array(y_test)

# --------------------------
# 3️⃣ Scale features
# --------------------------
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

# --------------------------
# 4️⃣ Dataset
# --------------------------
from sklearn.model_selection import train_test_split

# Split train into train + validation
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=SEED, stratify=y_train
)

class TabDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_ds = TabDataset(X_tr, y_tr)
val_ds   = TabDataset(X_val, y_val)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

# --------------------------
# 5️⃣ MLP Model
# --------------------------
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, dropout=0.3):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x, return_feature=False):
        h = x
        for layer in self.model[:-1]:
            h = layer(h)
        if return_feature:
            return h
        logits = self.model[-1](h).squeeze(1)
        return logits

model = MLP(input_dim=X_train.shape[1], hidden_dim=HIDDEN_DIM, dropout=0.3).to(DEVICE)
criterion = nn.BCEWithLogitsLoss()
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

# --------------------------
# 6️⃣ Train loop with validation + early stopping
# --------------------------
best_val_loss = float('inf')
patience = 5
counter = 0

for epoch in range(EPOCHS):
    # Train
    model.train()
    epoch_loss = 0.0
    for x_batch, y_batch in train_loader:
        x_batch, y_batch = x_batch.to(DEVICE), y_batch.to(DEVICE)
        optimizer.zero_grad()
        logits = model(x_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * x_batch.size(0)
    epoch_loss /= len(train_loader.dataset)

    # Validation
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for x_batch, y_batch in val_loader:
            x_batch, y_batch = x_batch.to(DEVICE), y_batch.to(DEVICE)
            logits = model(x_batch)
            loss = criterion(logits, y_batch)
            val_loss += loss.item() * x_batch.size(0)
    val_loss /= len(val_loader.dataset)

    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {epoch_loss:.4f} | Val Loss: {val_loss:.4f}")

    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_model.pth")
        counter = 0
    else:
        counter += 1
        if counter >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break

# Load best model
model.load_state_dict(torch.load("best_model.pth"))

# --------------------------
# 7️⃣ Feature extraction
# --------------------------
def extract_feature(model, X, batch_size=64):
    model.eval()
    feats = []
    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            x_batch = torch.tensor(X[i:i+batch_size], dtype=torch.float32).to(DEVICE)
            h = model(x_batch, return_feature=True)
            feats.append(h.cpu().numpy())
    return np.vstack(feats)

Z_train = extract_feature(model, X_train)
Z_test  = extract_feature(model, X_test)

# --------------------------
# 8️⃣ Save features
# --------------------------
pd.DataFrame(Z_train).to_csv(os.path.join(OUTPUT_DIR, "EmbedSup4-2_main_train.csv"), index=True)
pd.DataFrame(Z_test).to_csv(os.path.join(OUTPUT_DIR, "EmbedSup4-2-main_test.csv"), index=True)

print("Feature extraction complete!")
print(f"Z_train shape: {Z_train.shape}")
print(f"Z_test  shape: {Z_test.shape}")
print(f"Saved CSV to {OUTPUT_DIR}")



In [ ]:
# DNN
# ============================
# ✅ Full Imports
# ============================
import os
import math
import random
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_curve, auc, precision_recall_curve
from sklearn.model_selection import StratifiedKFold

from tensorflow.keras.models import Sequential, model_from_json
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam


# ============================
# 🔒 Fix random seed
# ============================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["TF_DETERMINISTIC_OPS"] = "1"


# -----------------------------
# Utility functions
# -----------------------------
def categorical_probas_to_classes(p):
    return np.argmax(p, axis=1)

def calculate_performance(test_num, pred_y, labels):
    tp = fp = tn = fn = 0
    for i in range(test_num):
        if labels[i] == 1:
            if pred_y[i] == 1: tp += 1
            else: fn += 1
        else:
            if pred_y[i] == 0: tn += 1
            else: fp += 1

    acc = (tp + tn) / test_num
    precision = tp / (tp + fp + 1e-6)
    npv = tn / (tn + fn + 1e-6)
    sensitivity = tp / (tp + fn + 1e-6)
    specificity = tn / (tn + fp + 1e-6)
    mcc = (tp * tn - fp * fn) / (
        math.sqrt((tp + fp)*(tp + fn)*(tn + fp)*(tn + fn)) + 1e-6
    )
    f1 = 2 * tp / (2 * tp + fp + fn + 1e-6)

    return acc, precision, npv, sensitivity, specificity, mcc, f1, tp, tn, fp, fn


# -----------------------------
# Model (DNN)
# -----------------------------
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam

def get_GRU_model(input_dim, out_dim):
    model = Sequential([
        Dense(256, activation='relu', input_shape=(input_dim,),
              kernel_initializer='he_normal'),
        BatchNormalization(),
        Dropout(0.95),

        Dense(128, activation='relu',
              kernel_initializer='he_normal'),
        BatchNormalization(),
        Dropout(0.9),

        Dense(64, activation='relu',
              kernel_initializer='he_normal'),
        BatchNormalization(),
        Dropout(0.9),

        Dense(out_dim, activation='softmax')
    ])

    model.compile(
        optimizer=Adam(learning_rate=1e-3),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# -----------------------------
# Load data
# -----------------------------
train_path = '/kaggle/working/EmbedSup4-2_main_train.csv'
test_path  = '/kaggle/working/EmbedSup4-2-main_test.csv'
train_df = pd.read_csv(train_path)
test_df  = pd.read_csv(test_path)

X_train = train_df.iloc[:, 1:].values
X_test  = test_df.iloc[:, 1:].values

y_train = [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
                  
y_test = [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
y_train = np.array(y_train)
y_test  = np.array(y_test)



# -----------------------------
# Normalize (ถูกหลัก)
# -----------------------------
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

input_dim = X_train.shape[1]
out_dim = len(np.unique(y_train))


# -----------------------------
# Cross-validation
# -----------------------------
save_dir = f'./DNN_seed{SEED}/'
os.makedirs(save_dir, exist_ok=True)

sepscores_cnn = []

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss', factor=0.2, patience=5, min_lr=1e-6
)
early_stop = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)
callbacks = [reduce_lr, early_stop]

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)

# ===== ADD (for ProbFeat only) =====
oof_train_prob = np.zeros((len(X_train), out_dim))
test_prob_folds = []

for i, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    print(f"\n===== Fold {i} =====")

    model = get_GRU_model(input_dim, out_dim)

    X_tr  = X_train[tr_idx]
    X_val = X_train[val_idx]
    y_tr  = to_categorical(y_train[tr_idx], out_dim)
    y_val = to_categorical(y_train[val_idx], out_dim)

    model.fit(
        X_tr, y_tr,
        validation_data=(X_val, y_val),
        epochs=50,
        batch_size=8,
        callbacks=callbacks,
        verbose=1
    )

    # ===== validation (เดิม) =====
    y_pred_prob = model.predict(X_val)
    y_pred = np.argmax(y_pred_prob, axis=1)

    acc, precision, npv, sensitivity, specificity, mcc, f1, tp, tn, fp, fn = \
        calculate_performance(len(y_pred), y_pred, y_train[val_idx])

    fpr, tpr, _ = roc_curve(y_train[val_idx], y_pred_prob[:, 1])
    roc_auc = auc(fpr, tpr)

    precision_curve, recall_curve, _ = precision_recall_curve(
        y_train[val_idx], y_pred_prob[:, 1]
    )
    au_pr = auc(recall_curve, precision_curve)

    sepscores_cnn.append(
        [acc, precision, npv, sensitivity, specificity,
         mcc, f1, roc_auc, au_pr, tp, tn, fp, fn]
    )

    # ===== ADD ONLY =====
    oof_train_prob[val_idx] = y_pred_prob
    test_prob_folds.append(model.predict(X_test))

    # Save model (เดิม)
    with open(f"{save_dir}/DNN_fold{i}.json", "w") as f:
        f.write(model.to_json())
    model.save_weights(f"{save_dir}/DNN_fold{i}.weights.h5")


cv_df = pd.DataFrame(
    sepscores_cnn,
    columns=['acc','precision','npv','sensitivity','specificity',
             'mcc','f1','roc_auc','au_pr','tp','tn','fp','fn']
)
cv_df.to_csv(f"{save_dir}/results_CV_DNNEmbedSup64-main.csv", index=False)


# -----------------------------
# Independent test (เดิมทั้งหมด)
# -----------------------------
sepscores_ind = []

for i in range(len(sepscores_cnn)):
    with open(f"{save_dir}/DNN_fold{i}.json", "r") as f:
        loaded_model = model_from_json(f.read())

    loaded_model.load_weights(f"{save_dir}/DNN_fold{i}.weights.h5")
    loaded_model.compile(
        loss='categorical_crossentropy',
        optimizer=Adam(),
        metrics=['accuracy']
    )

    y_score = loaded_model.predict(X_test)
    y_class = np.argmax(y_score, axis=1)

    acc, precision, npv, sensitivity, specificity, mcc, f1, tp, tn, fp, fn = \
        calculate_performance(len(y_class), y_class, y_test)

    fpr, tpr, _ = roc_curve(y_test, y_score[:, 1])
    roc_auc = auc(fpr, tpr)

    precision_curve, recall_curve, _ = precision_recall_curve(
        y_test, y_score[:, 1]
    )
    au_pr = auc(recall_curve, precision_curve)

    sepscores_ind.append(
        [acc, precision, npv, sensitivity, specificity,
         mcc, f1, roc_auc, au_pr, tp, tn, fp, fn]
    )

    print(f"Fold {i} test: MCC={mcc:.4f}, AUC={roc_auc:.4f}, AU_PR={au_pr:.4f}")

test_df_result = pd.DataFrame(
    sepscores_ind,
    columns=['acc','precision','npv','sensitivity','specificity',
             'mcc','f1','roc_auc','au_pr','tp','tn','fp','fn']
)
test_df_result.to_csv(f"{save_dir}/results_Independent_DNNEmbedSup64-main.csv", index=False)


# =============================
# ✅ SAVE ProbFeat (NEW ONLY)
# =============================
prob_train_df = pd.DataFrame(
    oof_train_prob,
    columns=[f'Prob_class{i}' for i in range(out_dim)]
)
prob_train_df.to_csv(f"{save_dir}/ProbFeat_train.csv", index=False)

avg_test_prob = np.mean(test_prob_folds, axis=0)

prob_test_df = pd.DataFrame(
    avg_test_prob,
    columns=[f'Prob_class{i}' for i in range(out_dim)]
)
prob_test_df.to_csv(f"{save_dir}/ProbFeat_test.csv", index=False)

print("✅ Saved ProbFeat_train.csv and ProbFeat_test.csv")
